1. Setup

In [1]:
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError
import pandas as pd

client = EntsoePandasClient(api_key="7308a46a-2e78-4bf2-a552-3295649e49e3")

eu_countries = ['AT','BE','BG','CZ','FR','GR','HU','SI','ES']

# Period for solar profile and prices: April–October 2024
sol_start   = pd.Timestamp("2024-04-01", tz="UTC")
sol_end     = pd.Timestamp("2024-11-01", tz="UTC")
price_start = sol_start
price_end   = sol_end

2. Hourly solar profile for a country

In [2]:
def get_hourly_solar_profile(country_code, start, end):
    """
    Return hourly solar profile (average MW by hour-of-day, 0..23)
    for given country and period. If no solar is reported, return None.
    """
    df = client.query_generation(country_code, start=start, end=end)
    if df.empty:
        return None

    # Keep only "Actual Aggregated" columns
    if isinstance(df.columns, pd.MultiIndex):
        mask = df.columns.get_level_values(1) == "Actual Aggregated"
        df_agg = df.loc[:, mask].copy()
        df_agg.columns = df_agg.columns.get_level_values(0)  # tech names only
    else:
        df_agg = df.copy()

    solar = df_agg.get("Solar")
    if solar is None:
        return None

    # Convert 15-min to hourly average MW (lowercase 'h' for your pandas)
    solar_hourly = solar.resample("1h").mean()

    solar_df = solar_hourly.to_frame(name="MW")
    solar_df["hour"] = solar_df.index.hour

    # Average solar output by hour of day (0–23)
    hourly_profile = solar_df.groupby("hour")["MW"].mean()
    return hourly_profile

3. Compute strong solar hours per preselected country

In [ ]:
strong_hours_by_country = {}
failed_solar = []

for cc in eu_countries:
    print(f"Computing solar profile for {cc} ...", end=" ")

    hp = get_hourly_solar_profile(cc, sol_start, sol_end)
    if hp is None or hp.empty:
        print("no solar profile")
        failed_solar.append(cc)
        continue

    max_avg = hp.max()
    threshold = 0.01 * max_avg          # solar hours > 1% of peak
    threshold_strong = 0.20 * max_avg   # strong hours > 20% of peak

    solar_hours = hp[hp > threshold].index.tolist()
    strong_solar_hours = hp[hp > threshold_strong].index.tolist()

    strong_hours_by_country[cc] = strong_solar_hours

    print(f"OK | Solar hours (>1% peak): {solar_hours} | Strong (>20%): {strong_solar_hours}")

print("\nCountries without usable solar profile:", failed_solar)
print("\nStrong solar hours by country:")
for cc, hrs in strong_hours_by_country.items():
    print(cc, ":", hrs)

Computing solar profile for AT ... OK | Solar hours (>1% peak): [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] | Strong (>20%): [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Computing solar profile for BE ... OK | Solar hours (>1% peak): [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] | Strong (>20%): [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Computing solar profile for BG ... OK | Solar hours (>1% peak): [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] | Strong (>20%): [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Computing solar profile for CZ ... OK | Solar hours (>1% peak): [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21] | Strong (>20%): [8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Computing solar profile for FR ... OK | Solar hours (>1% peak): [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21] | Strong (>20%): [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
Computing solar profile for GR ... OK | Solar hours (>1% peak): [7, 8, 9, 10, 11, 12,

4. Compute price statistics using those strong hours

In [ ]:
rows = []
failed_price = []

for cc in eu_countries:
    print(f"Processing prices for {cc} ...", end=" ")

    # Skip if we have no strong hours for this country
    if cc not in strong_hours_by_country:
        print("no strong-hours definition (skipping)")
        failed_price.append(cc)
        continue

    strong_hours = strong_hours_by_country[cc]

    try:
        prices = client.query_day_ahead_prices(
            country_code=cc,
            start=price_start,
            end=price_end
        )
    except NoMatchingDataError:
        print("no price data")
        failed_price.append(cc)
        continue
    except Exception as e:
        print(f"error: {e}")
        failed_price.append(cc)
        continue

    if prices.empty:
        print("empty price series")
        failed_price.append(cc)
        continue

    prices = prices.to_frame(name="price_eur_per_mwh")
    prices["hour"] = prices.index.hour

    strong_prices = prices[prices["hour"].isin(strong_hours)].copy()
    rest_prices   = prices[~prices["hour"].isin(strong_hours)].copy()

    if strong_prices.empty or rest_prices.empty:
        print("insufficient data in one of the subsets")
        failed_price.append(cc)
        continue

    strong_mean   = strong_prices["price_eur_per_mwh"].mean()
    strong_median = strong_prices["price_eur_per_mwh"].median()
    rest_mean     = rest_prices["price_eur_per_mwh"].mean()
    rest_median   = rest_prices["price_eur_per_mwh"].median()

    rows.append({
        "country": cc,
        "strong_mean_EUR_MWh":   strong_mean,
        "strong_median_EUR_MWh": strong_median,
        "rest_mean_EUR_MWh":     rest_mean,
        "rest_median_EUR_MWh":   rest_median,
        "n_strong_points":       len(strong_prices),
        "n_rest_points":         len(rest_prices),
    })

    print("ok")

result = pd.DataFrame(rows)

Processing prices for AT ... ok
Processing prices for BE ... ok
Processing prices for BG ... ok
Processing prices for CZ ... ok
Processing prices for FR ... ok
Processing prices for DE ... no price data
Processing prices for GR ... ok
Processing prices for HU ... ok
Processing prices for SI ... ok
Processing prices for ES ... ok


5. View summary for your preselected EU countries